# Day 3 — Semantic Retrieval

Goal: given a free-text trial idea, return the most similar historical trials + their outcomes.
Saves `../src/trials.faiss` and `../src/trials_meta.parquet` (the Day 4 API loads both).

Run top to bottom with **Shift+Enter**. Pick your **Python** kernel if asked. Read curriculum section 8 first.

Note: the first cell downloads a ~80MB embedding model the first time — give it a minute.

## Step 1 — load data + the embedding model

In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

df = pd.read_csv("../data/trials_features.csv").dropna(subset=["brief_summary"]).reset_index(drop=True)
df["label"] = (df["overall_status"].str.upper() == "COMPLETED").astype(int)
print("Trials with a summary:", len(df))

model = SentenceTransformer("all-MiniLM-L6-v2")   # fast, strong generalist
print("Model loaded.")

## Step 2 — embed all trial summaries

This is the slow cell. ~1–3 min for tens of thousands of trials on CPU. `normalize_embeddings=True` lets us use inner product as cosine similarity.

In [ ]:
emb = model.encode(
    df["brief_summary"].tolist(),
    show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True
)
print("Embedding matrix:", emb.shape)   # (n_trials, 384)

## Step 3 — build the FAISS index and save

In [ ]:
index = faiss.IndexFlatIP(emb.shape[1])   # inner product on normalized vectors = cosine
index.add(emb)
faiss.write_index(index, "../src/trials.faiss")
df.to_parquet("../src/trials_meta.parquet")
print("Indexed", index.ntotal, "trials -> ../src/trials.faiss")

## Step 4 — the search function

In [ ]:
def search(query, k=5):
    q = model.encode([query], normalize_embeddings=True)
    scores, idx = index.search(q, k)
    out = df.iloc[idx[0]][["nct_id", "phase", "overall_status", "label", "brief_summary"]].copy()
    out["similarity"] = scores[0]
    return out

res = search("Phase 2 trial of a GLP-1 agonist for type 2 diabetes in adults")
res

## Step 5 — sanity-check retrieval on a few queries

Eyeball these: do the returned trials actually match the query's topic? That's your evidence the retrieval works.

In [ ]:
for q in [
    "immunotherapy for advanced melanoma",
    "randomized trial of a statin to lower cholesterol",
    "vaccine study in healthy adult volunteers",
]:
    print("="*80)
    print("QUERY:", q)
    r = search(q, k=3)
    for _, row in r.iterrows():
        print(f"  [{row['similarity']:.2f}] {row['nct_id']} ({row['overall_status']}): {row['brief_summary'][:120]}...")
    print(f"  -> historical completion rate of these {len(r)}: {r['label'].mean():.2f}")

## Step 6 (optional, high-signal) — generalist vs biomedical embedder bake-off

Research finding: on short clinical text, general models often beat specialized biomedical ones. Test it and report which won — a memorable interview point. Skip if short on time (downloads another model).

In [ ]:
# OPTIONAL — comment out if short on time.
bio = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO")
sample = df.sample(min(3000, len(df)), random_state=1).reset_index(drop=True)
bio_emb = bio.encode(sample["brief_summary"].tolist(), convert_to_numpy=True,
                     normalize_embeddings=True, show_progress_bar=True)
bio_index = faiss.IndexFlatIP(bio_emb.shape[1]); bio_index.add(bio_emb)

def bio_search(query, k=3):
    q = bio.encode([query], normalize_embeddings=True)
    s, i = bio_index.search(q, k)
    return sample.iloc[i[0]][["nct_id", "brief_summary"]].assign(similarity=s[0])

print("Biomedical model results for 'immunotherapy for advanced melanoma':")
for _, row in bio_search("immunotherapy for advanced melanoma").iterrows():
    print(f"  [{row['similarity']:.2f}] {row['brief_summary'][:120]}...")